In [ ]:
# Assignment 7 — NLP I
# Important: the PDF requires tickets generated by a starter notebook using a roll number/fingerprint, which wasn't supplied here. This notebook is self-contained and uses illustrative IT-support tickets. Substitute your generated tickets if you need the exact dataset required by your instructor.

In [ ]:
import re, string, math
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from collections import Counter
for resource in ['punkt', 'punkt_tab', 'stopwords', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'wordnet', 'omw-1.4']:
    try: nltk.download(resource, quiet=True)
    except Exception: pass

In [ ]:
# Example tickets and preprocessing

In [ ]:
tickets = [
 "My laptop is not connecting to Wi-Fi. Please reset the network adapter.",
 "The printer stopped working after the latest update; error 95.5 appeared.",
 "I cannot access email at helpdesk@example.com. Please call 555-123-4567.",
 "The VPN connection fails on version 2.0.1 when I open the portal.",
 "A password reset link https://support.example.com/reset is not loading.",
 "The monitor flickers and the keyboard is responding slowly.",
 "Please install Python 3.11 and update the graphics driver.",
 "My account is locked after several login attempts.",
 "The Teams meeting audio is missing, but video works.",
 "The shared folder permissions need to be changed urgently.",
 "Laptop battery drains quickly and the device gets hot.",
 "I receive a blue-screen error whenever I launch the application."
]
TEST_LINES = [
 "Wi-Fi isn't working; version 2.0.1 is installed.",
 "Contact me at support@example.com or https://example.com, 555-123-4567.",
 "The measurement is 95.5 and the release is 3.1.4."
]
stop_words = set(stopwords.words('english'))
def process_ticket(text):
    sentences = sent_tokenize(text)
    tokens = word_tokenize(text)
    punctuation_tokens = [t for t in tokens if all(ch in string.punctuation for ch in t)]
    stops = [t for t in tokens if t.lower() in stop_words]
    final_tokens = [t.lower() for t in tokens if t.lower() not in stop_words and t not in punctuation_tokens]
    return {'sentences': sentences, 'tokens': tokens, 'punctuation': punctuation_tokens,
            'stopwords': stops, 'unique_tokens': set(tokens), 'final_tokens': final_tokens}

processed = [process_ticket(t) for t in tickets]
summary = pd.DataFrame([{
    'ticket': i+1, 'sentences': len(p['sentences']), 'tokens': len(p['tokens']),
    'punctuation_tokens': len(p['punctuation']), 'stopwords': len(p['stopwords']),
    'unique_tokens': len(p['unique_tokens']), 'final_tokens': len(p['final_tokens'])
} for i,p in enumerate(processed)])
summary.loc[len(summary)] = ['TOTAL', summary.sentences.sum(), summary.tokens.sum(),
    summary.punctuation_tokens.sum(), summary.stopwords.sum(), '', summary.final_tokens.sum()]
display(summary)

In [ ]:
chosen = processed[0]
print("Tokens:", chosen['tokens'])
print("Punctuation tokens:", chosen['punctuation'])
print("Stopwords:", chosen['stopwords'])
print("Final tokens:", chosen['final_tokens'])

In [ ]:
# Regex extraction and replacements

In [ ]:
raw_text = "\\n".join(tickets + TEST_LINES)
print("Words >5 letters:", re.findall(r'\\b[A-Za-z]{6,}\\b', raw_text))
print("Numbers/version-like:", re.findall(r'\\b\\d+(?:\\.\\d+)*\\b', raw_text))
print("Capitalized words:", re.findall(r'\\b[A-Z][a-z]*\\b', raw_text))
print("Words beginning with vowel:", re.findall(r'\\b[aeiouAEIOU][A-Za-z]*\\b', raw_text))

email_pattern = r'\\b[\\w.+-]+@[\\w.-]+\\.[A-Za-z]{2,}\\b'
url_pattern = r'https?://\\S+|www\\.\\S+'
phone_pattern = r'(?<!\\w)(?:\\+?\\d[\\d .()-]{7,}\\d)(?!\\w)'
redacted = re.sub(email_pattern, '<EMAIL>', raw_text)
redacted = re.sub(url_pattern, '<URL>', redacted)
redacted = re.sub(phone_pattern, '<PHONE>', redacted)
print(redacted)

In [ ]:
# Custom tokenization

In [ ]:
pattern = r'https?://\\S+|www\\.\\S+|[\\w.+-]+@[\\w.-]+\\.[A-Za-z]{2,}|\\+?\\d[\\d .()-]{7,}\\d|\\d+(?:\\.\\d+)+|[A-Za-z]+(?:[-’'][A-Za-z]+)*|[^\\w\\s]'
custom_tokenizer = RegexpTokenizer(pattern)
def my_tokenize(text):
    return custom_tokenizer.tokenize(text)

for line in TEST_LINES:
    print("\\nLine:", line)
    print("Custom:", my_tokenize(line))
    print("NLTK:", word_tokenize(line))

In [ ]:
# Typical differences: the custom pattern keeps URLs, email addresses, phone numbers, version strings and hyphenated words intact; standard word_tokenize() may split some of these into several tokens. The custom tokenizer also follows the priorities in its regex pattern.

In [ ]:
# Stemming and POS-aware lemmatization

In [ ]:
final_unique = sorted(set(token for p in processed for token in p['final_tokens']))
porter = PorterStemmer()
lancaster = LancasterStemmer()
# A simple suffix rule for demonstration; selected suffix based on 'working' and 'connecting'.
regexp_stemmer = RegexpStemmer(r'(ing|ed|s)$', min=4)
lemmatizer = WordNetLemmatizer()

def penn_to_wordnet(tag):
    if tag.startswith('J'): return 'a'
    if tag.startswith('V'): return 'v'
    if tag.startswith('N'): return 'n'
    if tag.startswith('R'): return 'r'
    return 'n'

rows = []
for word in final_unique:
    pos = penn_to_wordnet(pos_tag([word])[0][1])
    outputs = [porter.stem(word), lancaster.stem(word), regexp_stemmer.stem(word),
               lemmatizer.lemmatize(word, pos)]
    if len(set(outputs)) > 1:
        rows.append([word] + outputs)
diff_table = pd.DataFrame(rows, columns=['word','Porter','Lancaster','Regexp','POS lemma'])
display(diff_table)

In [ ]:
# The simple RegexpStemmer suffix rule was chosen after observing dataset words such as working and connecting. It is a heuristic and may remove suffixes incorrectly. Lancaster is often more aggressive than Porter. POS-aware lemmatization can return a dictionary word and uses part-of-speech information.

In [ ]:
# Corpus statistics, Zipf plot and bigrams

In [ ]:
from nltk.util import ngrams
S1 = [token for p in processed for token in p['tokens']]
S2 = [t for t in S1 if t not in processed[0]['punctuation']] # corrected below globally
S2 = [t for t in S1 if not all(ch in string.punctuation for ch in t)]
S3 = [token for p in processed for token in p['final_tokens']]
S4 = [porter.stem(t) for t in S3]

def corpus_stats(name, corpus):
    return {'corpus':name, 'total_tokens':len(corpus),
            'unique_tokens':len(set(corpus)),
            'TTR':len(set(corpus))/len(corpus) if corpus else 0}
display(pd.DataFrame([corpus_stats('S1',S1),corpus_stats('S2',S2),
                      corpus_stats('S3',S3),corpus_stats('S4',S4)]))

print("Top 10 S1:", Counter(S1).most_common(10))
print("Top 10 S3:", Counter(S3).most_common(10))
print("Top 5 bigrams S1:", Counter(ngrams(S1,2)).most_common(5))
print("Top 5 bigrams S3:", Counter(ngrams(S3,2)).most_common(5))

import matplotlib.pyplot as plt
freq = sorted(Counter(S3).values(), reverse=True)
ranks = np.arange(1, len(freq)+1)
plt.figure()
plt.loglog(ranks, freq, marker='o')
plt.xlabel('Rank (log scale)')
plt.ylabel('Frequency (log scale)')
plt.title('Zipf plot for S3')
plt.grid(True)
plt.show()

In [ ]:
# Interpretation
# (a) TTR typically declines as token count grows and changes after stemming because stemming merges variants into shared forms, reducing the number of unique tokens relative to the total.
#
# (b) A custom tokenizer may preserve a version number such as 2.0.1 as one token, whereas word_tokenize() can split it.
#
# (c) Stemming can be undesirable when it truncates a word into a non-word or merges words with different meanings.
#
# (d) POS-aware lemmatization can preserve a meaningful dictionary form—for example, correctly lemmatizing a verb's inflected form based on its POS rather than applying a suffix rule blindly.